# Beyond Random Sampling: How to Prioritize Agent Trajectories Without an LLM
Production AI agents generate thousands of conversation trajectories every day, but only a small fraction are truly useful for improving the system. The challenge is not just identifying failures, but finding the interactions that expose meaningful behavioral weaknesses — subtle policy violations, inefficient tool usage, redundant reasoning steps, or stalled conversations that standard sampling methods often miss. Random sampling wastes review budget on routine successes, while simple heuristics like trajectory length tend to surface only obvious breakdowns.

In this article, we reproduce the core idea from a recent paper by DigitalOcean using a small synthetic customer-support environment. We generate 13 trajectories across five behavioral archetypes — clean completions, misalignment, stagnation, disengagement, and satisfaction — using GPT-4o-mini, then score each conversation using seven deterministic behavioral signals extracted directly from messages and execution logs. The results mirror the paper’s findings: signal-based sampling surfaces the most informative trajectories more reliably than either random or length-based selection. More importantly, it catches subtle inefficiencies that other methods overlook — including a completed support task where the agent redundantly called the same verification tool twice with identical inputs, technically succeeding while still wasting execution steps.

## Setting up the dependencies

In [1]:
import os
import json
import random
import string
import textwrap
from dataclasses import dataclass, field
from typing import Optional
from openai import OpenAI
from getpass import getpass

In [ ]:
OPENAI_API_KEY = getpass('Enter OpenAI API Key: ')
MODEL = "gpt-4o-mini"

client = OpenAI(api_key=OPENAI_API_KEY)

random.seed(42)

## Data Structures
We first define lightweight data structures to represent agent conversations and execution traces. Message captures each interaction step — including user inputs, assistant responses, and tool calls — while Trajectory stores the full conversation history, tool execution logs, system errors, and task outcome. These structured objects form the foundation for extracting deterministic behavioral signals directly from trajectory data.

In [ ]:
@dataclass
class Message:
    role: str          # "user" | "assistant" | "tool"
    content: str
    tool_name: Optional[str] = None    # populated when role == "tool"
    tool_input: Optional[dict] = None  # the arguments passed to the tool


@dataclass
class Trajectory:
    id: str
    messages: list[Message] = field(default_factory=list)
    tool_calls: list[dict] = field(default_factory=list)   # execution log
    env_errors: list[str] = field(default_factory=list)    # system-level errors
    task_completed: bool = False

## Generating Synthetic Trajectories via OpenAI
Next, we build a synthetic trajectory generation pipeline to simulate realistic customer-support interactions. Instead of relying on manually collected conversations, we define a set of scenario templates representing different behavioral archetypes — including clean completions, user-agent misalignment, stagnation loops, disengagement, satisfaction, tool failures, and redundant tool usage. Each scenario contains task instructions along with behavioral constraints that guide the model toward producing specific interaction patterns.

The generate_trajectory() function uses the OpenAI API to create structured conversations in JSON format. The system prompt enforces strict rules around task completion status, tool success behavior, conversation length, and user reactions, ensuring that the generated trajectories remain consistent and easy to analyze later. After generation, the JSON output is parsed into Trajectory objects containing messages, tool calls, environment errors, and completion metadata. Finally, build_trajectory_dataset() iterates through all predefined scenarios to create a small but behaviorally diverse dataset that we later use for deterministic signal extraction and sampling experiments.

In [ ]:
SCENARIOS = [
    # ── Clean paths (should complete, all tools succeed) ──────────────────────
    {
        "label": "clean_password_reset",
        "task": "Help a user reset their password.",
        "inject": None,
        "expect_completed": True,
        "expect_tool_success": True,
    },
    {
        "label": "clean_order_tracking",
        "task": "Help a user track the status of their recent order.",
        "inject": None,
        "expect_completed": True,
        "expect_tool_success": True,
    },
    {
        "label": "clean_plan_upgrade",
        "task": "Help a user upgrade from the Basic to the Pro plan.",
        "inject": None,
        "expect_completed": True,
        "expect_tool_success": True,
    },
    # ── Misalignment (user rephrases / corrects the agent) ────────────────────
    {
        "label": "misalignment_order",
        "task": "Help a user find out why their order was delayed.",
        "inject": (
            "The agent keeps asking for the order ID even though the user already "
            "provided it. The user gets frustrated and corrects the agent multiple times, "
            "saying things like 'I already gave you my order number' or 'as I mentioned, "
            "it is 98765'. The task eventually completes but only after several corrections."
        ),
        "expect_completed": True,
        "expect_tool_success": True,
    },
    {
        "label": "misalignment_refund",
        "task": "Help a user request a refund for a damaged item.",
        "inject": (
            "The agent misunderstands the user's issue and tries to process an exchange "
            "instead of a refund. The user has to clarify multiple times: 'No, I want a "
            "refund, not an exchange' and 'that's not what I asked for'. Eventually the "
            "agent understands and completes the correct task."
        ),
        "expect_completed": True,
        "expect_tool_success": True,
    },
    # ── Stagnation (agent loops on the same response) ─────────────────────────
    {
        "label": "stagnation_subscription",
        "task": "Help a user cancel their subscription.",
        "inject": (
            "The agent gets stuck in a loop. After the user provides their account email, "
            "the agent keeps repeating the exact same response asking for account details. "
            "The user provides the information again and again but the agent loops. "
            "The task is NOT completed. Use nearly identical assistant messages to show the loop."
        ),
        "expect_completed": False,
        "expect_tool_success": False,
    },
    {
        "label": "stagnation_billing",
        "task": "Help a user update their billing address.",
        "inject": (
            "The agent acknowledges receiving the new address but then asks for it again "
            "in the very next turn. This repeats 3 times. The user grows increasingly "
            "frustrated. The task is NOT completed. The assistant messages should be "
            "near-identical to each other to demonstrate stagnation."
        ),
        "expect_completed": False,
        "expect_tool_success": False,
    },
    # ── Disengagement (user abandons before completion) ───────────────────────
    {
        "label": "disengagement_support",
        "task": "Help a user troubleshoot why they cannot log in to their account.",
        "inject": (
            "The agent's suggestions don't work and the user gets increasingly frustrated. "
            "Eventually the user says something like 'this is getting nowhere, I'll just "
            "contact support directly' or 'forget it, I'll try another way' and leaves. "
            "The task is NOT completed. Include an env_error like 'auth_service_timeout'."
        ),
        "expect_completed": False,
        "expect_tool_success": False,
    },
    {
        "label": "disengagement_shipping",
        "task": "Help a user change the shipping address on an order that has already shipped.",
        "inject": (
            "The agent cannot actually change the address since the order is in transit. "
            "It keeps offering alternatives the user doesn't want. The user eventually "
            "says 'never mind, I'll just deal with it' and ends the conversation. "
            "The task is NOT completed."
        ),
        "expect_completed": False,
        "expect_tool_success": False,
    },
    # ── Satisfaction (clean completion with explicit user confirmation) ────────
    {
        "label": "satisfaction_password",
        "task": "Help a user reset a forgotten password.",
        "inject": (
            "Everything works smoothly. At the end the user says something like "
            "'perfect, thank you so much' or 'that worked, I'm all set'. "
            "The task is completed successfully."
        ),
        "expect_completed": True,
        "expect_tool_success": True,
    },
    {
        "label": "satisfaction_address",
        "task": "Help a user update the email address on their account.",
        "inject": (
            "The agent handles it efficiently. The user confirms at the end with "
            "something like 'great, that's exactly what I needed' or 'looks good, thanks'. "
            "The task is completed successfully."
        ),
        "expect_completed": True,
        "expect_tool_success": True,
    },
    # ── Tool failure (tool call returns an error) ──────────────────────────────
    {
        "label": "tool_failure_lookup",
        "task": "Help a user check the status of a return they submitted.",
        "inject": (
            "The lookup_return tool fails with an error on the first call (success: false). "
            "The agent tries a fallback tool which also fails. The task is NOT completed. "
            "Include an env_error like 'returns_service_unavailable'."
        ),
        "expect_completed": False,
        "expect_tool_success": False,
    },
    # ── Tool loop (same tool called twice with identical inputs) ───────────────
    {
        "label": "tool_loop_verify",
        "task": "Help a user verify their identity to unlock a locked account.",
        "inject": (
            "The agent calls the verify_identity tool with the same parameters twice "
            "in a row because it doesn't register the first result. Both calls have "
            "identical inputs. The task eventually completes but the redundant call "
            "wasted a step. In tool_calls, include two consecutive identical tool entries."
        ),
        "expect_completed": True,
        "expect_tool_success": True,
    },
]


def generate_trajectory(scenario: dict) -> Trajectory:
    """
    Call the OpenAI API to produce a realistic agent–user conversation
    for the given scenario, then parse it into a Trajectory object.

    The model returns JSON so we can deterministically extract signals later.
    """
    task   = scenario["task"]
    inject = scenario.get("inject")
    expect_completed     = scenario.get("expect_completed", True)
    expect_tool_success  = scenario.get("expect_tool_success", True)

    system_prompt = textwrap.dedent(f"""
        You are simulating a realistic customer-support agent conversation.
        Return ONLY a valid JSON object with this exact schema — no markdown, no extra text:

        {{
          "task_completed": true | false,
          "messages": [
            {{"role": "user" | "assistant", "content": "<text>"}},
            ...
          ],
          "tool_calls": [
            {{"tool": "<name>", "input": {{"key": "value"}}, "success": true | false}},
            ...
          ],
          "env_errors": ["<error string>", ...]
        }}

        Hard rules — you MUST follow these exactly:
        1. task_completed MUST be {str(expect_completed).lower()}.
        2. Tool call success values MUST reflect the scenario:
           - If expect_tool_success is {str(expect_tool_success).lower()}, set success accordingly.
           - For failure scenarios: at least one tool call must have "success": false.
           - For success scenarios: ALL tool calls must have "success": true.
        3. 5–10 messages total (user and assistant alternating, starting with user).
        4. Include 2–4 tool calls with realistic names (lookup_order, verify_identity,
           send_email, update_record, cancel_subscription, lookup_return, etc.).
        5. env_errors: empty list [] unless the scenario description mentions a specific error.
        6. Make the conversation feel realistic — use natural, varied language.
           Avoid robotic or repetitive phrasing UNLESS the scenario explicitly calls for it
           (e.g. stagnation scenarios should have near-identical assistant messages).
        7. For misalignment/correction scenarios: the user messages MUST contain natural
           correction phrases such as "as I mentioned", "I already told you", "that's not
           what I asked", "no, I want a refund not an exchange", etc.
        8. For disengagement scenarios: the final user message MUST contain an abandonment
           phrase such as "never mind", "forget it", "I'll just deal with it",
           "this is getting nowhere", or similar.
        9. For satisfaction scenarios: the final user message MUST contain a satisfaction
           phrase such as "perfect", "thank you", "that worked", "I'm all set", etc.
    """).strip()

    user_prompt = f"Task: {task}"
    if inject:
        user_prompt += f"\n\nScenario instructions: {inject}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.7,
    )

    raw = response.choices[0].message.content.strip()

    # Strip accidental markdown fences if the model adds them
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    data = json.loads(raw)

    traj_id = "traj_" + "".join(random.choices(string.ascii_lowercase, k=6))

    messages = [
        Message(role=m["role"], content=m["content"])
        for m in data.get("messages", [])
    ]

    return Trajectory(
        id=traj_id,
        messages=messages,
        tool_calls=data.get("tool_calls", []),
        env_errors=data.get("env_errors", []),
        task_completed=data.get("task_completed", False),
    )


def build_trajectory_dataset(n_per_scenario: int = 1) -> list[Trajectory]:
    """
    Generate `n_per_scenario` trajectories per scenario template.
    Default is 1 per scenario (13 scenarios → 13 trajectories).
    Increase n_per_scenario to build a larger pool.
    """
    dataset: list[Trajectory] = []
    for scenario in SCENARIOS:
        for _ in range(n_per_scenario):
            try:
                traj = generate_trajectory(scenario)
                dataset.append(traj)
                print(f"  ✓ {traj.id}  [{scenario['label']:<30}]"
                      f"  completed={traj.task_completed}"
                      f"  msgs={len(traj.messages)}"
                      f"  tools={len(traj.tool_calls)}")
            except Exception as exc:
                print(f"  ✗ {scenario['label']} — generation failed: {exc}")
    return dataset

## Signal Detectors
This section implements the deterministic signal detectors used to analyze agent behavior directly from trajectory logs. Instead of relying on another LLM to evaluate conversations, each detector is implemented as a lightweight rule-based function that operates on message content, tool execution logs, or environment metadata. We first define phrase libraries representing behaviors such as user corrections, frustration, abandonment, and satisfaction, then use simple string matching to identify these patterns in conversations.

The detectors are grouped into three categories. Interaction signals capture conversational issues such as misalignment, stagnation, disengagement, and explicit user satisfaction. Execution signals analyze tool behavior, including failed tool calls and redundant tool loops where the same tool is invoked multiple times with identical inputs. Finally, environment signals capture infrastructure-level issues such as API failures or service outages. Together, these functions create a fully deterministic behavioral scoring layer that can surface problematic or informative trajectories without any additional annotation or model inference overhead.

In [ ]:
REPHRASE_PHRASES = [
    # explicit rephrase markers
    "what i mean is", "let me rephrase", "let me clarify", "to be clear",
    "what i meant", "i meant to say", "actually i", "actually, i",
    "to clarify", "what i'm asking", "what i was asking",
    "i said", "i already said", "i already told you", "i already provided",
    "i already gave you", "as i mentioned", "like i said",
    "no, i want", "no i want", "not that", "different question",
    "let me try again", "i'm asking about", "my question was",
    # frustrated repetition patterns
    "i just told you", "i just said", "i just gave you",
]

CORRECTION_PHRASES = [
    # direct corrections
    "that's wrong", "that's not right", "that is wrong", "that is not right",
    "incorrect", "you misunderstood", "you're misunderstanding",
    "that's not what i asked", "that's not what i said",
    "no, that's not", "no that's not", "wrong answer", "wrong information",
    "you got it wrong", "not what i wanted", "not what i asked for",
    # softer but still corrective
    "that's not quite", "not exactly", "that's not correct",
    "you have the wrong", "the wrong", "that's incorrect",
    "i think you misread", "you seem to have misunderstood",
    "that's not my", "no, my",
]

SATISFACTION_PHRASES = [
    # explicit thanks / confirmation
    "thank you", "thanks", "thank you so much", "thanks so much",
    "that worked", "it worked", "that did it", "that fixed it",
    "perfect", "great", "excellent", "wonderful", "fantastic",
    "exactly what i needed", "just what i needed", "that's exactly",
    "got it", "awesome", "resolved", "all set", "all good",
    "confirmed", "yes that's correct", "that's it", "that's the one",
    # task closure signals
    "problem is solved", "issue is resolved", "everything is working",
    "that's all i needed", "i'm all set", "great, that's done",
    "looks good", "you've been helpful", "very helpful",
]

ABANDONMENT_PHRASES = [
    # explicit abandonment
    "never mind", "nevermind", "forget it", "forget about it",
    "this isn't helping", "this is not helping", "this isn't useful",
    "i'll figure it out", "i'll sort it out", "i'll handle it myself",
    "let me try something else", "giving up", "i give up",
    "not helpful", "useless", "waste of time",
    # frustrated exit signals
    "i'll just", "i'll contact", "i'll call", "i'll email",
    "this is frustrating", "this is pointless", "going in circles",
    "i'm done", "i'll try another way", "forget the whole thing",
    "this is getting nowhere", "not going to work",
]


def _user_messages(traj: Trajectory) -> list[str]:
    return [m.content.lower() for m in traj.messages if m.role == "user"]


def _assistant_messages(traj: Trajectory) -> list[str]:
    return [m.content.lower() for m in traj.messages if m.role == "assistant"]


def _phrase_hit(text: str, phrases: list[str]) -> bool:
    return any(p in text for p in phrases)


# ── 3a. Interaction Signals ───────────────────────────────────────────────────

def detect_misalignment(traj: Trajectory) -> bool:
    """
    User rephrased or corrected the agent → the agent misunderstood something.
    """
    for msg in _user_messages(traj):
        if _phrase_hit(msg, REPHRASE_PHRASES + CORRECTION_PHRASES):
            return True
    return False


def detect_stagnation(traj: Trajectory, sim_threshold: float = 0.85) -> bool:
    """
    Agent repeated itself → conversation is stuck.
    We use token-overlap (Jaccard) as a lightweight similarity proxy.
    """
    assistant_msgs = _assistant_messages(traj)
    if len(assistant_msgs) < 2:
        return False

    def jaccard(a: str, b: str) -> float:
        sa, sb = set(a.split()), set(b.split())
        if not sa or not sb:
            return 0.0
        return len(sa & sb) / len(sa | sb)

    for i in range(len(assistant_msgs) - 1):
        if jaccard(assistant_msgs[i], assistant_msgs[i + 1]) >= sim_threshold:
            return True
    return False


def detect_disengagement(traj: Trajectory) -> bool:
    """
    User abandoned the conversation before the task completed.
    """
    if traj.task_completed:
        return False
    for msg in _user_messages(traj):
        if _phrase_hit(msg, ABANDONMENT_PHRASES):
            return True
    return False


def detect_satisfaction(traj: Trajectory) -> bool:
    """
    User explicitly confirmed success.
    """
    for msg in _user_messages(traj):
        if _phrase_hit(msg, SATISFACTION_PHRASES):
            return True
    return False


# ── 3b. Execution Signals ────────────────────────────────────────────────────

def detect_tool_failure(traj: Trajectory) -> bool:
    """
    Any tool call that did not succeed is a failure signal.
    """
    return any(not tc.get("success", True) for tc in traj.tool_calls)


def detect_tool_loop(traj: Trajectory) -> bool:
    """
    The same tool was called twice in a row with identical inputs → loop.
    """
    calls = traj.tool_calls
    for i in range(len(calls) - 1):
        if (calls[i]["tool"] == calls[i + 1]["tool"] and
                calls[i].get("input") == calls[i + 1].get("input")):
            return True
    return False


# ── 3c. Environment Signals ──────────────────────────────────────────────────

def detect_env_errors(traj: Trajectory) -> bool:
    """
    Rate limits, context overflow, API errors — useful for diagnosis,
    but excluded from the training score (system constraints ≠ agent decisions).
    """
    return len(traj.env_errors) > 0

## Score & Rank
Once the behavioral signals are extracted, we assign weights to each one to estimate how informative a trajectory is for human review. More severe or diagnostically useful behaviors — such as user disengagement, stagnation loops, or redundant tool calls — receive higher weights, while positive outcomes like explicit user satisfaction receive only a small contribution. Environment-level errors are tracked for debugging purposes but excluded from the optimization score since they reflect system constraints rather than agent decision-making.

The score_trajectory() function runs all signal detectors on a trajectory, records which behaviors were triggered, and computes a final score by summing the corresponding signal weights. Each result is stored in a ScoredTrajectory object containing the original trajectory, detected signals, and overall score. Finally, rank_trajectories() sorts the dataset from highest to lowest score, allowing us to prioritize the most behaviorally informative conversations for annotation and analysis.

In [ ]:
SIGNAL_WEIGHTS = {
    "misalignment":   2.0,   # user had to correct/rephrase
    "stagnation":     2.5,   # agent looped on the same response
    "disengagement":  3.0,   # user gave up
    "tool_failure":   2.0,   # a tool call returned failure
    "tool_loop":      2.5,   # identical back-to-back tool calls
    # satisfaction is a *positive* signal — mild weight, still worth reviewing
    "satisfaction":   0.5,
    # env errors: informative for diagnosis, not for agent improvement
    "env_error":      0.0,
}


@dataclass
class ScoredTrajectory:
    trajectory: Trajectory
    signals: dict[str, bool]
    score: float


def score_trajectory(traj: Trajectory) -> ScoredTrajectory:
    signals = {
        "misalignment":  detect_misalignment(traj),
        "stagnation":    detect_stagnation(traj),
        "disengagement": detect_disengagement(traj),
        "satisfaction":  detect_satisfaction(traj),
        "tool_failure":  detect_tool_failure(traj),
        "tool_loop":     detect_tool_loop(traj),
        "env_error":     detect_env_errors(traj),
    }
    score = sum(SIGNAL_WEIGHTS[k] for k, fired in signals.items() if fired)
    return ScoredTrajectory(trajectory=traj, signals=signals, score=score)


def rank_trajectories(dataset: list[Trajectory]) -> list[ScoredTrajectory]:
    scored = [score_trajectory(t) for t in dataset]
    return sorted(scored, key=lambda s: s.score, reverse=True)

## Sampling Strategies (for comparison)
To evaluate whether signal-based selection is actually more effective, we implement three different trajectory sampling strategies. The first is random sampling, which selects conversations uniformly at random and serves as a simple baseline. The second is length-based sampling, which prioritizes the longest conversations under the assumption that longer trajectories are more likely to contain failures or inefficiencies.

Finally, we implement signal-based sampling, where trajectories are first scored using the deterministic behavioral signals from the previous section and then ranked from most to least informative. The top-ranked trajectories are selected for review. This setup allows us to directly compare whether lightweight behavioral scoring surfaces more useful conversations than traditional heuristics like randomness or conversation length.

In [ ]:
def random_sample(dataset: list[Trajectory], k: int) -> list[Trajectory]:
    return random.sample(dataset, min(k, len(dataset)))


def length_based_sample(dataset: list[Trajectory], k: int) -> list[Trajectory]:
    sorted_by_len = sorted(dataset, key=lambda t: len(t.messages), reverse=True)
    return sorted_by_len[:k]


def signal_based_sample(ranked: list[ScoredTrajectory], k: int) -> list[ScoredTrajectory]:
    return ranked[:k]

## Informativeness Evaluation (ground-truth proxy)
In the original paper, trajectories are manually labeled by human annotators as either informative or non-informative for improving the agent. Since this experiment uses synthetic data, we simulate that labeling process using a deterministic proxy. A trajectory is considered informative if it contains any meaningful behavioral issue — such as misalignment, stagnation, disengagement, tool failures, or redundant tool loops — or if the task failed to complete successfully.

The is_informative() function implements this logic by first scoring the trajectory and then checking whether any non-trivial signals were triggered. Positive-only signals like user satisfaction and environment-level issues are excluded from this decision since they are less useful for optimization. The informativeness_rate() function then measures the quality of a sampled subset by calculating the proportion of trajectories that are considered informative. This gives us a simple evaluation metric for comparing random, length-based, and signal-based sampling strategies.

In [ ]:
def is_informative(traj: Trajectory) -> bool:
    """
    Proxy for human 'worth reviewing' label.
    """
    scored = score_trajectory(traj)
    non_trivial = {k for k, v in scored.signals.items()
                   if v and k not in ("satisfaction", "env_error")}
    return bool(non_trivial) or not traj.task_completed


def informativeness_rate(sample: list[Trajectory]) -> float:
    if not sample:
        return 0.0
    return sum(is_informative(t) for t in sample) / len(sample)

## Running the main demo
main() function first generates a synthetic dataset of customer-support conversations, scores each trajectory using the deterministic behavioral detectors, and ranks them based on weighted signal scores. It then compares three sampling strategies — random sampling, length-based filtering, and signal-based ranking — to evaluate which method surfaces the most informative conversations for human review.

To ground this in practice, we generated 13 synthetic customer-support trajectories across behavioral archetypes such as clean completions, misalignment, stagnation, disengagement, satisfaction, tool failures, and redundant tool usage using GPT-4o-mini. Each trajectory was analyzed using seven deterministic signals extracted directly from message content and execution logs. The output shows that signal-based sampling surfaced the top 6 most informative trajectories with a 100% informativeness rate, outperforming both length-based filtering (83.3%) and random sampling (66.7%). This closely mirrors the trend reported in the original τ-bench paper, where behavioral signal ranking consistently outperformed heuristic baselines.

The detailed inspection step further highlights why the approach works. Instead of only surfacing obviously broken conversations, the ranking process also identified subtle inefficiencies that standard heuristics would likely miss. One of the top-ranked trajectories was technically successful — the user’s account was unlocked — but the agent unnecessarily called the verify_identity tool twice with identical inputs before proceeding. Although the task completed successfully, the redundant step revealed an execution inefficiency that still matters for optimization, latency, and production cost. The final signal frequency breakdown also provides a quick diagnostic overview of the dataset by showing how often each behavioral issue appeared across all generated trajectories.

In [2]:
def print_separator(title: str = "") -> None:
    width = 65
    if title:
        pad = (width - len(title) - 2) // 2
        print("\n" + "─" * pad + f" {title} " + "─" * pad)
    else:
        print("\n" + "─" * width)


def display_scored(st: ScoredTrajectory, rank: int) -> None:
    t = st.trajectory
    fired = [k for k, v in st.signals.items() if v]
    print(f"  #{rank:>2}  {t.id}  score={st.score:.1f}"
          f"  completed={t.task_completed}"
          f"  msgs={len(t.messages)}")
    print(f"        signals: {', '.join(fired) if fired else '(none)'}")


def main():
    print_separator("SIGNAL-BASED TRAJECTORY SAMPLING")
    print("""
Goal: Given a pool of agent trajectories, surface the top-k most
informative ones for human review — without using an LLM scorer.

Method: Compute lightweight behavioral signals from raw trajectory
data using deterministic rules, then rank by weighted signal score.
""")

    # ── Step 1: Generate dataset ─────────────────────────────────────────────
    print_separator("Step 1 · Generating trajectories via OpenAI")
    # 13 scenario templates × 1 each = 13 trajectories by default.
    # Increase n_per_scenario to build a larger pool (costs more API calls).
    dataset = build_trajectory_dataset(n_per_scenario=1)
    print(f"\nTotal trajectories generated: {len(dataset)}")

    if not dataset:
        print("No trajectories generated. Check your API key and try again.")
        return

    # ── Step 2: Score all trajectories ───────────────────────────────────────
    print_separator("Step 2 · Scoring with behavioral signals")
    ranked = rank_trajectories(dataset)

    print("\nAll trajectories (highest signal first):\n")
    for i, st in enumerate(ranked, 1):
        display_scored(st, i)

    # ── Step 3: Compare sampling strategies ──────────────────────────────────
    print_separator("Step 3 · Comparing sampling strategies")

    k = min(6, len(dataset))   # review top-k; ~46% of dataset mirrors paper's ratio

    rand_sample     = random_sample(dataset, k)
    length_sample   = length_based_sample(dataset, k)
    signal_sample_r = signal_based_sample(ranked, k)
    signal_sample   = [st.trajectory for st in signal_sample_r]

    rand_rate   = informativeness_rate(rand_sample)
    length_rate = informativeness_rate(length_sample)
    signal_rate = informativeness_rate(signal_sample)

    print(f"""
  Sample size : {k} trajectories (out of {len(dataset)})

  Strategy              Informative rate
  ─────────────────────────────────────
  Random sampling       {rand_rate * 100:.1f}%
  Length-based          {length_rate * 100:.1f}%
  Signal-based          {signal_rate * 100:.1f}%   ← this approach
""")

    # ── Step 4: Inspect top picks ────────────────────────────────────────────
    print_separator("Step 4 · Top signal-based picks (detail view)")

    for st in signal_sample_r[:3]:
        t = st.trajectory
        print(f"\n  Trajectory : {t.id}")
        print(f"  Score      : {st.score:.1f}")
        print(f"  Completed  : {t.task_completed}")
        print(f"  Signals    : {[k for k,v in st.signals.items() if v]}")
        print(f"  Tool calls : {[tc['tool'] for tc in t.tool_calls]}")
        print(f"  Messages   :")
        for m in t.messages:
            prefix = "  U:" if m.role == "user" else "  A:"
            snippet = m.content[:90].replace("\n", " ")
            print(f"    {prefix} {snippet}{'…' if len(m.content) > 90 else ''}")

    # ── Step 5: Signal breakdown ──────────────────────────────────────────────
    print_separator("Step 5 · Signal frequency across full dataset")

    all_signals = ["misalignment", "stagnation", "disengagement",
                   "satisfaction", "tool_failure", "tool_loop", "env_error"]

    print()
    for sig in all_signals:
        count = sum(1 for st in ranked if st.signals.get(sig))
        bar   = "█" * count + "░" * (len(ranked) - count)
        note  = " (diagnosis only, excluded from score)" if sig == "env_error" else ""
        print(f"  {sig:<18} {bar}  {count}/{len(ranked)}{note}")

    # ── Step 6: Key takeaways ────────────────────────────────────────────────
    print_separator("Key Takeaways")
    print("""
  1. Random sampling wastes annotation budget on routine, clean conversations.

  2. Length-based filtering catches obvious failures (long = stuck) but
     misses subtle issues in conversations where the agent technically succeeded.

  3. Signal-based sampling surfaces subtle problems — policy violations,
     inefficient tool use, unnecessary steps — even in completed tasks.

  4. The whole framework is deterministic: no LLM overhead, runs always-on
     in a production pipeline directly on execution logs.

  5. Environment signals (rate limits, context overflow) are useful for
     diagnosis but excluded from the training score — they reflect system
     constraints, not agent decision quality.
""")
    print_separator()


if __name__ == "__main__":
    main()


─────────────── SIGNAL-BASED TRAJECTORY SAMPLING ───────────────

Goal: Given a pool of agent trajectories, surface the top-k most
informative ones for human review — without using an LLM scorer.

Method: Compute lightweight behavioral signals from raw trajectory
data using deterministic rules, then rank by weighted signal score.


────────── Step 1 · Generating trajectories via OpenAI ──────────
  ✓ traj_qahftr  [clean_password_reset          ]  completed=True  msgs=10  tools=2
  ✓ traj_xckafn  [clean_order_tracking          ]  completed=True  msgs=10  tools=2
  ✓ traj_afqofp  [clean_plan_upgrade            ]  completed=True  msgs=9  tools=2
  ✓ traj_vausie  [misalignment_order            ]  completed=True  msgs=10  tools=2
  ✓ traj_yiccwp  [misalignment_refund           ]  completed=True  msgs=9  tools=2
  ✓ traj_usnzjo  [stagnation_subscription       ]  completed=False  msgs=9  tools=2
  ✓ traj_vqwpsb  [stagnation_billing            ]  completed=False  msgs=9  tools=2
  ✓ traj_fhcg